# Base vs Random Scenario Divergence Analysis

Analyze how random scenarios diverge from the base deterministic scenario using assumptions_sam.json.

**Hypothesis**: Variance in interest rates causes geometric compounding effects that significantly reduce wealth accumulation in random scenarios compared to the base scenario.

**Scope**: Compare all accounts (savings + trad 401k + roth 401k + roth IRA + HSA) at key ages (60, 75, 80, 90, 110).

In [1]:
import os
import json
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Configure display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# Django setup
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django

django.setup()

from pages.base_scenario import BaseScenario
from pages.random_scenario import RandomScenario

print("✓ Imports complete")

✓ Imports complete


## Step 1: Load Assumptions

In [2]:
# Load assumptions from JSON file
with open("assumptions_sam.json", "r") as f:
    assumptions = json.load(f)

print("=" * 80)
print("SCENARIO ANALYSIS WITH ASSUMPTIONS_SAM.JSON")
print("=" * 80)
print("\nAssumptions Summary:")
print(f"  Birthdate: {assumptions['birthdate']}")
print(
    f"  Retirement age: {assumptions['retirement_age_yrs']}y {assumptions['retirement_age_mos']}m"
)
print(f"  Base savings: ${assumptions['base_savings']:,.0f}")
print(
    f"  Monthly savings: ${assumptions['base_saved_per_mo']:,.0f} (+{assumptions['base_savings_per_yr_increase']}%/year)"
)
print(
    f"  Base interest rates: RF={assumptions['base_rf_interest_per_yr']}%, MKT={assumptions['base_mkt_interest_per_yr']}%"
)
print(f"  Inflation: {assumptions['base_inflation_per_yr']}%")
print("  Retirement accounts:")
for acct in assumptions["retirement_accounts"]:
    status = "ACTIVE" if acct["is_active"] else "inactive"
    print(
        f"    - {acct['retirement_type']}: ${acct['base_retirement']:,.0f} ({status})"
    )

SCENARIO ANALYSIS WITH ASSUMPTIONS_SAM.JSON

Assumptions Summary:
  Birthdate: 12/22/1986
  Retirement age: 60y 0m
  Base savings: $160,000
  Monthly savings: $1,000 (+3.0%/year)
  Base interest rates: RF=3.25%, MKT=6.5%
  Inflation: 2.75%
  Retirement accounts:
    - traditional_401k: $160,000 (ACTIVE)
    - roth_401k: $160,000 (ACTIVE)
    - roth_ira: $56,000 (inactive)
    - hsa: $25,000 (ACTIVE)


## Step 2: Run Base Scenario (Deterministic)

In [3]:
print("\n" + "=" * 80)
print("RUNNING BASE SCENARIO (deterministic)")
print("=" * 80)

base_scenario = BaseScenario(assumptions)
base_df = base_scenario.create_base_df()

print(f"\n✓ Base scenario complete: {len(base_df)} months")
print(
    f"  Age range: {base_df['age_yrs'].min()}y {base_df['age_mos'].min()}m to {base_df['age_yrs'].max()}y {base_df['age_mos'].max()}m"
)

# Define account columns to track
account_cols = ["savings_account", "traditional_401k", "roth_401k", "roth_ira", "hsa"]
var_account_cols = ["var_" + col for col in account_cols]

print(
    f"\nBase scenario column check: {all(col in base_df.columns for col in account_cols)}"
)


RUNNING BASE SCENARIO (deterministic)

✓ Base scenario complete: 893 months
  Age range: 39y 0m to 113y 11m

Base scenario column check: True


In [4]:
# Extract base scenario at key ages
key_ages = [60, 75, 80, 90, 110]
base_milestones = {}

print("\nBASE SCENARIO KEY MILESTONES:")
print("-" * 80)

for target_age in key_ages:
    rows = base_df[base_df["age_yrs"] == target_age]
    if len(rows) > 0:
        row = rows.iloc[0]
        base_milestones[target_age] = {
            "savings_account": row["savings_account"],
            "traditional_401k": row["traditional_401k"],
            "roth_401k": row["roth_401k"],
            "roth_ira": row["roth_ira"],
            "hsa": row["hsa"],
        }
        total = sum(base_milestones[target_age].values())
        base_milestones[target_age]["total"] = total

        print(f"\nAge {target_age}:")
        print(f"  Savings account:    ${row['savings_account']:>14,.0f}")
        print(f"  Trad 401k:          ${row['traditional_401k']:>14,.0f}")
        print(f"  Roth 401k:          ${row['roth_401k']:>14,.0f}")
        print(f"  Roth IRA:           ${row['roth_ira']:>14,.0f}")
        print(f"  HSA:                ${row['hsa']:>14,.0f}")
        print("  " + "-" * 40)
        print(f"  TOTAL:              ${total:>14,.0f}")

print("\n" + "=" * 80)


BASE SCENARIO KEY MILESTONES:
--------------------------------------------------------------------------------

Age 60:
  Savings account:    $       523,974
  Trad 401k:          $     1,076,316
  Roth 401k:          $     1,092,636
  Roth IRA:           $       526,943
  HSA:                $       174,527
  ----------------------------------------
  TOTAL:              $     3,394,395

Age 75:
  Savings account:    $        85,422
  Trad 401k:          $     2,449,756
  Roth 401k:          $     2,495,355
  Roth IRA:           $       774,660
  HSA:                $       398,583
  ----------------------------------------
  TOTAL:              $     6,203,775

Age 80:
  Savings account:    $       188,530
  Trad 401k:          $     2,551,167
  Roth 401k:          $     3,239,756
  Roth IRA:           $     1,005,753
  HSA:                $       517,486
  ----------------------------------------
  TOTAL:              $     7,502,693

Age 90:
  Savings account:    $       619,998
 

## Step 3: Generate Random Scenarios

In [5]:
print("\n" + "=" * 80)
print("GENERATING RANDOM SCENARIOS (stochastic)")
print("=" * 80)

num_scenarios = 10
random_dfs = {}

for i in range(num_scenarios):
    print(f"\nGenerating scenario {i+1}/{num_scenarios}...", end="", flush=True)
    random_scenario = RandomScenario(base_scenario)
    random_dfs[i + 1] = random_scenario.create_full_df()
    print(" ✓")

print(f"\n✓ All {num_scenarios} scenarios complete")
print(f"  Each scenario: {len(random_dfs[1])} months")

# Verify variable columns exist
print(
    f"\n  Variable column check: {all(col in random_dfs[1].columns for col in var_account_cols)}"
)


GENERATING RANDOM SCENARIOS (stochastic)

Generating scenario 1/10... ✓

Generating scenario 2/10... ✓

Generating scenario 3/10... ✓

Generating scenario 4/10... ✓

Generating scenario 5/10... ✓

Generating scenario 6/10... ✓

Generating scenario 7/10... ✓

Generating scenario 8/10... ✓

Generating scenario 9/10... ✓

Generating scenario 10/10... ✓

✓ All 10 scenarios complete
  Each scenario: 893 months

  Variable column check: True


## Step 4: Compare All Accounts at Key Ages

In [6]:
print("\n" + "=" * 80)
print("SIDE-BY-SIDE COMPARISON: BASE vs RANDOM SCENARIOS")
print("=" * 80)

# For each key age, show base + all scenarios
for target_age in key_ages:
    print(f"\n{'='*80}")
    print(f"AGE {target_age}")
    print(f"{'='*80}")

    # Create comparison table
    comparison_data = []

    # Base scenario row
    if target_age in base_milestones:
        base_row = base_milestones[target_age]
        comparison_data.append(
            {
                "Scenario": "BASE",
                "Savings": base_row["savings_account"],
                "Trad 401k": base_row["traditional_401k"],
                "Roth 401k": base_row["roth_401k"],
                "Roth IRA": base_row["roth_ira"],
                "HSA": base_row["hsa"],
                "TOTAL": base_row["total"],
            }
        )

    # Random scenarios
    for scenario_num in range(1, num_scenarios + 1):
        random_df = random_dfs[scenario_num]
        rows = random_df[random_df["age_yrs"] == target_age]

        if len(rows) > 0:
            row = rows.iloc[0]
            total = (
                row["var_savings_account"]
                + row["var_traditional_401k"]
                + row["var_roth_401k"]
                + row["var_roth_ira"]
                + row["var_hsa"]
            )

            comparison_data.append(
                {
                    "Scenario": f"SCEN {scenario_num}",
                    "Savings": row["var_savings_account"],
                    "Trad 401k": row["var_traditional_401k"],
                    "Roth 401k": row["var_roth_401k"],
                    "Roth IRA": row["var_roth_ira"],
                    "HSA": row["var_hsa"],
                    "TOTAL": total,
                }
            )

    # Create DataFrame and display
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))

    # Calculate divergence from base
    if len(comparison_df) > 1:
        base_total = comparison_df.iloc[0]["TOTAL"]
        print("\nDivergence from BASE (% difference in TOTAL):")
        for idx in range(1, len(comparison_df)):
            scenario_total = comparison_df.iloc[idx]["TOTAL"]
            pct_diff = (
                ((scenario_total - base_total) / abs(base_total) * 100)
                if base_total != 0
                else 0
            )
            print(f"  {comparison_df.iloc[idx]['Scenario']}: {pct_diff:+.1f}%")

print("\n" + "=" * 80)


SIDE-BY-SIDE COMPARISON: BASE vs RANDOM SCENARIOS

AGE 60
Scenario    Savings    Trad 401k    Roth 401k   Roth IRA        HSA        TOTAL
    BASE 523,973.82 1,076,315.70 1,092,636.03 526,942.84 174,526.82 3,394,395.22
  SCEN 1 465,851.00   634,653.74   598,080.19 297,661.38 101,559.20 2,097,805.50
  SCEN 2 469,972.21   606,867.40   604,405.92 306,205.22  98,964.90 2,086,415.66
  SCEN 3 575,093.11   550,727.83   584,100.22 326,560.08 100,517.94 2,136,999.18
  SCEN 4 458,818.42   624,769.51   622,538.95 300,138.77  95,513.90 2,101,779.54
  SCEN 5 571,584.79   637,915.19   646,574.19 318,046.18  99,971.85 2,274,092.21
  SCEN 6 500,476.30   623,329.54   607,016.40 292,744.36 100,886.84 2,124,453.44
  SCEN 7 581,806.91   612,744.16   619,950.81 307,154.01  97,607.16 2,219,263.05
  SCEN 8 529,135.74   580,565.49   660,607.16 292,539.94 104,589.93 2,167,438.27
  SCEN 9 483,373.97   574,099.57   565,554.57 295,993.14  91,987.51 2,011,008.76
 SCEN 10 524,459.41   634,423.25   582,236.44 290,

## Step 5: Interest Rate Variations

In [7]:
print("\n" + "=" * 80)
print("INTEREST RATE VARIATIONS (first 5 years)")
print("=" * 80)

# Show interest rates for first 60 months
num_months_to_show = 60

print("\nBASE SCENARIO (deterministic rates):")
print(f"{'Month':<8} {'Age':<10} {'RF Rate':<12} {'MKT Rate':<12}")
print("-" * 42)
for idx in range(0, min(num_months_to_show, len(base_df))):
    row = base_df.iloc[idx]
    print(
        f"{idx:<8} {row['age_yrs']:.0f}y{row['age_mos']:02.0f}m   {row['yearly_rf_interest']:>10.2f}%  {row['yearly_mkt_interest']:>10.2f}%"
    )

# Show a couple random scenarios
for scenario_num in [1, 2]:
    print(f"\nSCENARIO {scenario_num} (stochastic rates):")
    print(f"{'Month':<8} {'Age':<10} {'RF Rate':<12} {'MKT Rate':<12}")
    print("-" * 42)
    random_df = random_dfs[scenario_num]
    for idx in range(0, min(num_months_to_show, len(random_df))):
        row = random_df.iloc[idx]
        print(
            f"{idx:<8} {row['age_yrs']:.0f}y{row['age_mos']:02.0f}m   {row['var_yearly_rf_interest']:>10.2f}%  {row['var_yearly_mkt_interest']:>10.2f}%"
        )

print("\n" + "=" * 80)


INTEREST RATE VARIATIONS (first 5 years)

BASE SCENARIO (deterministic rates):
Month    Age        RF Rate      MKT Rate    
------------------------------------------
0        39y05m         0.03%        0.07%
1        39y06m         0.03%        0.07%
2        39y07m         0.03%        0.07%
3        39y08m         0.03%        0.07%
4        39y09m         0.03%        0.07%
5        39y10m         0.03%        0.07%
6        39y11m         0.03%        0.07%
7        40y00m         0.03%        0.07%
8        40y01m         0.03%        0.07%
9        40y02m         0.03%        0.07%
10       40y03m         0.03%        0.07%
11       40y04m         0.03%        0.07%
12       40y05m         0.03%        0.07%
13       40y06m         0.03%        0.07%
14       40y07m         0.03%        0.07%
15       40y08m         0.03%        0.07%
16       40y09m         0.03%        0.07%
17       40y10m         0.03%        0.07%
18       40y11m         0.03%        0.07%
19       41y00

## Step 6: Compounding Effect Analysis

Calculate geometric vs. arithmetic returns to understand how volatility reduces wealth accumulation.

In [8]:
def calculate_compound_return(rates, label=""):
    """Calculate compound return from a series of annual rates (as percentages)."""
    if len(rates) == 0:
        return 0
    # Convert percentage rates to multipliers and compound
    multiplier = 1.0
    for rate in rates:
        multiplier *= 1 + rate / 100
    compound_return = (multiplier - 1) * 100  # Convert back to percentage
    return compound_return


print("\n" + "=" * 80)
print("COMPOUNDING EFFECT: GEOMETRIC vs ARITHMETIC RETURNS")
print("=" * 80)

# Calculate compounding over first 5, 10, 20 years (in months)
periods = [
    (60, "5 years"),
    (120, "10 years"),
    (240, "20 years"),
]

for months, label in periods:
    print(f"\n{'-'*80}")
    print(f"Over {label} ({months} months):")
    print(f"{'-'*80}")

    # Base scenario compounding
    base_rates = base_df.iloc[:months]["yearly_rf_interest"].values
    base_compound = calculate_compound_return(base_rates, "Base RF")
    base_avg_arithmetic = np.mean(base_rates)

    print("\nBASE SCENARIO:")
    print(f"  RF Rate (arithmetic mean):     {base_avg_arithmetic:>7.2f}%")
    print(f"  RF Rate (compound return):     {base_compound:>7.2f}%")
    print(
        f"  Difference (compound loss):    {base_compound - base_avg_arithmetic:>7.2f}%"
    )

    # Random scenarios compounding
    scenario_compounds = []
    for scenario_num in range(1, min(4, num_scenarios + 1)):  # Show first 3 scenarios
        random_df = random_dfs[scenario_num]
        scenario_rates = random_df.iloc[:months]["var_yearly_rf_interest"].values
        scenario_compound = calculate_compound_return(
            scenario_rates, f"Scenario {scenario_num}"
        )
        scenario_avg_arithmetic = np.mean(scenario_rates)
        scenario_compounds.append(scenario_compound)

        print(f"\nSCENARIO {scenario_num}:")
        print(f"  RF Rate (arithmetic mean):     {scenario_avg_arithmetic:>7.2f}%")
        print(f"  RF Rate (compound return):     {scenario_compound:>7.2f}%")
        print(
            f"  Difference (compound loss):    {scenario_compound - scenario_avg_arithmetic:>7.2f}%"
        )
        print(
            f"  vs Base compound return:       {scenario_compound - base_compound:>7.2f}%"
        )

print("\n" + "=" * 80)


COMPOUNDING EFFECT: GEOMETRIC vs ARITHMETIC RETURNS

--------------------------------------------------------------------------------
Over 5 years (60 months):
--------------------------------------------------------------------------------

BASE SCENARIO:
  RF Rate (arithmetic mean):        0.03%
  RF Rate (compound return):        1.97%
  Difference (compound loss):       1.94%

SCENARIO 1:
  RF Rate (arithmetic mean):        0.03%
  RF Rate (compound return):        1.85%
  Difference (compound loss):       1.82%
  vs Base compound return:         -0.12%

SCENARIO 2:
  RF Rate (arithmetic mean):        0.03%
  RF Rate (compound return):        1.80%
  Difference (compound loss):       1.77%
  vs Base compound return:         -0.17%

SCENARIO 3:
  RF Rate (arithmetic mean):        0.04%
  RF Rate (compound return):        2.14%
  Difference (compound loss):       2.11%
  vs Base compound return:          0.17%

------------------------------------------------------------------------

## Step 7: Identify Divergence Inflection Points

Track when random scenarios first fall significantly behind base scenario.

In [9]:
print("\n" + "=" * 80)
print("DIVERGENCE TRACKING: WHEN DO RANDOM SCENARIOS FALL BEHIND?")
print("=" * 80)

# Track total wealth month-by-month for each scenario
for scenario_num in range(1, num_scenarios + 1):
    print(f"\n{'='*80}")
    print(f"SCENARIO {scenario_num}")
    print(f"{'='*80}")

    random_df = random_dfs[scenario_num]

    # Calculate total wealth for each month in both base and scenario
    base_totals = (
        base_df["savings_account"]
        + base_df["traditional_401k"]
        + base_df["roth_401k"]
        + base_df["roth_ira"]
        + base_df["hsa"]
    )

    scenario_totals = (
        random_df["var_savings_account"]
        + random_df["var_traditional_401k"]
        + random_df["var_roth_401k"]
        + random_df["var_roth_ira"]
        + random_df["var_hsa"]
    )

    # Calculate percent difference month by month
    pct_diff = ((scenario_totals - base_totals) / base_totals.abs()) * 100

    # Find first month where divergence exceeds 5%, 10%, 20%
    thresholds = [5, 10, 20]
    for threshold in thresholds:
        diverged = pct_diff[pct_diff < -threshold]
        if len(diverged) > 0:
            first_month = diverged.index[0]
            first_row = random_df.iloc[first_month]
            base_row = base_df.iloc[first_month]
            print(f"\n  First month with >{threshold}% divergence:")
            print(
                f"    Month {first_month}: Age {first_row['age_yrs']:.0f}y {first_row['age_mos']:.0f}m"
            )
            print(
                f"    Base total:     ${base_row['savings_account'] + base_row['traditional_401k'] + base_row['roth_401k'] + base_row['roth_ira'] + base_row['hsa']:,.0f}"
            )
            print(
                f"    Scenario total: ${first_row['var_savings_account'] + first_row['var_traditional_401k'] + first_row['var_roth_401k'] + first_row['var_roth_ira'] + first_row['var_hsa']:,.0f}"
            )
            print(f"    Difference:     {pct_diff.iloc[first_month]:.1f}%")
        else:
            print(f"\n  >{threshold}% divergence: Never occurs")

    # Check if savings account ever goes negative
    if (random_df["var_savings_account"] < 0).any():
        first_negative = random_df[random_df["var_savings_account"] < 0].index[0]
        first_row = random_df.iloc[first_negative]
        print("\n  ⚠️  Savings account goes NEGATIVE:")
        print(
            f"    Month {first_negative}: Age {first_row['age_yrs']:.0f}y {first_row['age_mos']:.0f}m"
        )
        print(f"    Balance: ${first_row['var_savings_account']:,.0f}")
    else:
        final_savings = random_df.iloc[-1]["var_savings_account"]
        print("\n  ✓ Savings account stays positive")
        print(f"    Final balance (age 115): ${final_savings:,.0f}")

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)


DIVERGENCE TRACKING: WHEN DO RANDOM SCENARIOS FALL BEHIND?

SCENARIO 1

  First month with >5% divergence:
    Month 21: Age 41y 2m
    Base total:     $679,732
    Scenario total: $644,589
    Difference:     -5.2%

  First month with >10% divergence:
    Month 46: Age 43y 3m
    Base total:     $850,451
    Scenario total: $763,424
    Difference:     -10.2%

  First month with >20% divergence:
    Month 110: Age 48y 7m
    Base total:     $1,428,406
    Scenario total: $1,141,355
    Difference:     -20.1%

  ⚠️  Savings account goes NEGATIVE:
    Month 703: Age 98y 0m
    Balance: $-914

SCENARIO 2

  First month with >5% divergence:
    Month 27: Age 41y 8m
    Base total:     $718,682
    Scenario total: $680,977
    Difference:     -5.2%

  First month with >10% divergence:
    Month 49: Age 43y 6m
    Base total:     $872,599
    Scenario total: $780,341
    Difference:     -10.6%

  First month with >20% divergence:
    Month 102: Age 47y 11m
    Base total:     $1,344,304
  

## Step 8: Summary - Divergence Magnitude

In [10]:
print("\n" + "=" * 80)
print("DIVERGENCE SUMMARY TABLE")
print("=" * 80)

# Create a comprehensive summary table showing divergence at key ages
summary_rows = []

for target_age in [60, 75, 80, 90, 110]:
    if target_age in base_milestones:
        base_total = base_milestones[target_age]["total"]

        for scenario_num in range(1, num_scenarios + 1):
            random_df = random_dfs[scenario_num]
            rows = random_df[random_df["age_yrs"] == target_age]

            if len(rows) > 0:
                row = rows.iloc[0]
                scenario_total = (
                    row["var_savings_account"]
                    + row["var_traditional_401k"]
                    + row["var_roth_401k"]
                    + row["var_roth_ira"]
                    + row["var_hsa"]
                )
                pct_diff = (
                    ((scenario_total - base_total) / base_total * 100)
                    if base_total != 0
                    else 0
                )

                summary_rows.append(
                    {
                        "Age": target_age,
                        "Scenario": scenario_num,
                        "Base Total": base_total,
                        "Scenario Total": scenario_total,
                        "% Diff": pct_diff,
                    }
                )

summary_df = pd.DataFrame(summary_rows)

# Pivot to show ages as rows, scenarios as columns, with % divergence values
if len(summary_df) > 0:
    pivot_df = summary_df.pivot_table(index="Age", columns="Scenario", values="% Diff")
    print("\nPercent Divergence from Base Scenario (%):")
    print(pivot_df.round(1).to_string())

    # Calculate average divergence by age
    print("\n\nAverage Divergence by Age:")
    print("-" * 40)
    for age in [60, 75, 80, 90, 110]:
        age_data = summary_df[summary_df["Age"] == age]
        if len(age_data) > 0:
            avg_divergence = age_data["% Diff"].mean()
            min_divergence = age_data["% Diff"].min()
            max_divergence = age_data["% Diff"].max()
            print(
                f"Age {age:3d}: Average={avg_divergence:>7.1f}%, Range=[{min_divergence:>7.1f}%, {max_divergence:>7.1f}%]"
            )

print("\n" + "=" * 80)


DIVERGENCE SUMMARY TABLE

Percent Divergence from Base Scenario (%):
Scenario      1       2       3       4       5       6       7       8       9       10
Age                                                                                     
60        -38.20  -38.50  -37.00  -38.10  -33.00  -37.40  -34.60  -36.10  -40.80  -37.20
75        -70.40  -70.90  -68.70  -68.30  -63.50  -70.00  -66.80  -68.40  -72.70  -70.00
80        -79.00  -78.70  -76.60  -76.30  -72.60  -77.10  -74.60  -76.80  -80.30  -78.70
90        -91.70  -91.80  -89.40  -90.30  -87.00  -90.30  -87.70  -89.90  -92.70  -91.90
110      -111.10 -109.40 -107.20 -109.20 -105.50 -108.20 -105.20 -108.50 -110.80 -110.80


Average Divergence by Age:
----------------------------------------
Age  60: Average=  -37.1%, Range=[  -40.8%,   -33.0%]
Age  75: Average=  -69.0%, Range=[  -72.7%,   -63.5%]
Age  80: Average=  -77.1%, Range=[  -80.3%,   -72.6%]
Age  90: Average=  -90.3%, Range=[  -92.7%,   -87.0%]
Age 110: Average= -10

## Step 9: Rate Methodology Audit

Investigate the randomization parameters and check for systematic bias in how rates are applied.

In [11]:
from pages.random_scenario import (
    variance_1,
    variance_2,
    variance_3,
    RF_INTEREST_CHANGE_MOS,
)
from pages.base_scenario import MIN_CONSERVATIVE_RETIREMENT_RATE_PCT

assumed_rf = assumptions["base_rf_interest_per_yr"]  # 3.25%
assumed_mkt = assumptions["base_mkt_interest_per_yr"]  # 6.5%

print("=" * 80)
print("RANDOMIZATION PARAMETER AUDIT")
print("=" * 80)

print("\nVariance parameters:")
print(
    f"  variance_1 = {variance_1}  → std dev = {variance_1*100:.0f}% of mean (healthcare)"
)
print(
    f"  variance_2 = {variance_2}  → std dev = {variance_2*100:.0f}% of mean (bills, savings contributions)"
)
print(
    f"  variance_3 = {variance_3}  → std dev = {variance_3*100:.0f}% of mean (market interest rates)"
)

print("\nFor your assumptions:")
print(
    f"  RF rate:  mean={assumed_rf:.2f}%,  std dev={assumed_rf * variance_1:.2f}% (variance_1)"
)
print(
    f"           [RF actually uses a random walk of ±0.15% every {RF_INTEREST_CHANGE_MOS} months]"
)
print(
    f"  MKT rate: mean={assumed_mkt:.2f}%,  std dev={assumed_mkt * variance_3:.2f}% (variance_3)"
)
print(
    f"            → 1-sigma range: [{assumed_mkt - assumed_mkt*variance_3:.2f}%, {assumed_mkt + assumed_mkt*variance_3:.2f}%]"
)
print("            → Normal(6.5%, 9.75%) — can easily draw negative rates!")

print(f"\nMinimum conservative rate floor: {MIN_CONSERVATIVE_RETIREMENT_RATE_PCT}%")
print(
    "Upper cap: YES — rates are capped at the conservative baseline (one-sided cap!)"
)
print("Lower cap: NO — no floor other than what numpy produces")
print("\n⚠️  ONE-SIDED CAP CREATES SYSTEMATIC DOWNWARD BIAS:")
print("   When sampled > mean  → capped at mean (upside removed)")
print("   When sampled < mean  → used as-is (can go negative)")
print("   Result: E[applied rate] < assumed rate")

RANDOMIZATION PARAMETER AUDIT

Variance parameters:
  variance_1 = 0.1  → std dev = 10% of mean (healthcare)
  variance_2 = 0.5  → std dev = 50% of mean (bills, savings contributions)
  variance_3 = 1.5  → std dev = 150% of mean (market interest rates)

For your assumptions:
  RF rate:  mean=3.25%,  std dev=0.33% (variance_1)
           [RF actually uses a random walk of ±0.15% every 6 months]
  MKT rate: mean=6.50%,  std dev=9.75% (variance_3)
            → 1-sigma range: [-3.25%, 16.25%]
            → Normal(6.5%, 9.75%) — can easily draw negative rates!

Minimum conservative rate floor: 5.0%
Upper cap: YES — rates are capped at the conservative baseline (one-sided cap!)
Lower cap: NO — no floor other than what numpy produces

⚠️  ONE-SIDED CAP CREATES SYSTEMATIC DOWNWARD BIAS:
   When sampled > mean  → capped at mean (upside removed)
   When sampled < mean  → used as-is (can go negative)
   Result: E[applied rate] < assumed rate


In [12]:
print("\n" + "=" * 80)
print("QUANTIFYING THE ONE-SIDED CAP BIAS")
print("=" * 80)

# Simulate 100,000 draws to measure the actual expected rate vs. assumed
n_draws = 100_000
np.random.seed(42)

for mean_pct in [assumed_mkt, 7.0, 5.0]:
    std_pct = mean_pct * variance_3
    draws = np.random.normal(mean_pct, std_pct, n_draws)

    # Apply one-sided cap: cap at mean if above mean
    capped_draws = np.where(draws > mean_pct, mean_pct, draws)

    uncapped_mean = draws.mean()
    capped_mean = capped_draws.mean()
    bias = capped_mean - mean_pct

    print(f"\nMKT rate = {mean_pct:.1f}%  (std dev = {std_pct:.2f}%):")
    print(
        f"  Uncapped mean of draws:  {uncapped_mean:>7.3f}%  (should be ~{mean_pct:.1f}%)"
    )
    print(f"  Capped mean (applied):   {capped_mean:>7.3f}%")
    print(
        f"  Systematic bias:         {bias:>7.3f}%  ({bias/mean_pct*100:.1f}% of assumed rate)"
    )
    pct_draws_capped = (draws > mean_pct).mean() * 100
    pct_draws_negative = (capped_draws < 0).mean() * 100
    print(f"  Draws capped at top:     {pct_draws_capped:.1f}%")
    print(f"  Draws below zero:        {pct_draws_negative:.1f}%")


QUANTIFYING THE ONE-SIDED CAP BIAS

MKT rate = 6.5%  (std dev = 9.75%):
  Uncapped mean of draws:    6.509%  (should be ~6.5%)
  Capped mean (applied):     2.611%
  Systematic bias:          -3.889%  (-59.8% of assumed rate)
  Draws capped at top:     50.1%
  Draws below zero:        25.2%

MKT rate = 7.0%  (std dev = 10.50%):
  Uncapped mean of draws:    7.010%  (should be ~7.0%)
  Capped mean (applied):     2.821%
  Systematic bias:          -4.179%  (-59.7% of assumed rate)
  Draws capped at top:     50.0%
  Draws below zero:        25.2%

MKT rate = 5.0%  (std dev = 7.50%):
  Uncapped mean of draws:    4.989%  (should be ~5.0%)
  Capped mean (applied):     2.001%
  Systematic bias:          -2.999%  (-60.0% of assumed rate)
  Draws capped at top:     49.9%
  Draws below zero:        25.3%


In [13]:
print("\n" + "=" * 80)
print("ACTUAL vs ASSUMED RATES ACROSS SCENARIOS")
print("=" * 80)

print("\nComparing actual mean applied rates vs assumed rates:")
print(f"{'':20s} {'Assumed':>10s} {'Mean Actual':>12s} {'Bias':>10s}")
print("-" * 60)

for scenario_num in range(1, num_scenarios + 1):
    rdf = random_dfs[scenario_num]

    # Market rate (for retirement accounts)
    actual_mkt_mean = np.array(rdf["var_yearly_mkt_interest"].tolist()).mean() * 100
    mkt_bias = actual_mkt_mean - assumed_mkt

    # RF rate
    actual_rf_mean = np.array(rdf["var_yearly_rf_interest"].tolist()).mean() * 100
    rf_bias = actual_rf_mean - assumed_rf

    print(
        f"  Scen {scenario_num:2d}  RF:   {assumed_rf:>8.2f}%  {actual_rf_mean:>10.2f}%  {rf_bias:>+9.2f}%"
    )
    print(
        f"          MKT:  {assumed_mkt:>8.2f}%  {actual_mkt_mean:>10.2f}%  {mkt_bias:>+9.2f}%"
    )

# Summary
all_mkt_means = [
    np.array(random_dfs[s]["var_yearly_mkt_interest"].tolist()).mean() * 100
    for s in range(1, num_scenarios + 1)
]
all_rf_means = [
    np.array(random_dfs[s]["var_yearly_rf_interest"].tolist()).mean() * 100
    for s in range(1, num_scenarios + 1)
]

print(f"\nSummary across {num_scenarios} scenarios:")
print(
    f"  RF  assumed={assumed_rf:.2f}%, avg actual={np.mean(all_rf_means):.3f}%, avg bias={np.mean(all_rf_means)-assumed_rf:+.3f}%"
)
print(
    f"  MKT assumed={assumed_mkt:.2f}%, avg actual={np.mean(all_mkt_means):.3f}%, avg bias={np.mean(all_mkt_means)-assumed_mkt:+.3f}%"
)


ACTUAL vs ASSUMED RATES ACROSS SCENARIOS

Comparing actual mean applied rates vs assumed rates:
                        Assumed  Mean Actual       Bias
------------------------------------------------------------
  Scen  1  RF:       3.25%        3.56%      +0.31%
          MKT:      6.50%        3.19%      -3.31%
  Scen  2  RF:       3.25%        2.53%      -0.72%
          MKT:      6.50%        3.08%      -3.42%
  Scen  3  RF:       3.25%        3.08%      -0.17%
          MKT:      6.50%        3.11%      -3.39%
  Scen  4  RF:       3.25%        2.73%      -0.52%
          MKT:      6.50%        3.13%      -3.37%
  Scen  5  RF:       3.25%        3.48%      +0.23%
          MKT:      6.50%        3.09%      -3.41%
  Scen  6  RF:       3.25%        2.28%      -0.97%
          MKT:      6.50%        3.43%      -3.07%
  Scen  7  RF:       3.25%        4.77%      +1.52%
          MKT:      6.50%        3.00%      -3.50%
  Scen  8  RF:       3.25%        3.79%      +0.54%
          MKT

In [14]:
print("\n" + "=" * 80)
print(
    "DECOMPOSING: HOW MUCH OF THE GAP IS RATE BIAS vs VOLATILITY DRAG vs EXPENSE VARIANCE?"
)
print("=" * 80)

# Bills variance: variance_2=0.5 means std dev = 50% of mean
# For monthly bills of $1300, std dev = $650 — that's huge
# But bills are symmetric (normal), so no bias — only slightly more spending due to variance
# The savings increase also has variance_2=0.5 → std dev = 50% of savings amount

# Show the expense variance magnitudes
pre_retire_bills_base = base_df[base_df["age_yrs"] < 60]["base_bills"].mean()
print(f"\nExpense variance (variance_2 = {variance_2}, std dev = 50% of mean):")
print(f"  Pre-retirement avg monthly bills: ${pre_retire_bills_base:,.0f}")
print(f"  Std dev in bills: ${pre_retire_bills_base * variance_2:,.0f}")

savings_base_mean = base_df[base_df["age_yrs"] < 60]["savings_increase"].mean()
print(f"\n  Pre-retirement avg monthly savings: ${savings_base_mean:,.0f}")
print(f"  Std dev in savings contribution: ${savings_base_mean * variance_2:,.0f}")

print(f"\nMKT interest variance (variance_3 = {variance_3}, std dev = 150% of mean):")
print(f"  Assumed MKT rate: {assumed_mkt:.2f}%")
print(f"  Std dev: {assumed_mkt * variance_3:.2f}%")
print(
    f"  Normal({assumed_mkt:.1f}%, {assumed_mkt*variance_3:.2f}%) → ~50% of draws could be negative or zero!"
)

# Quantify downward bias from capping
# When Normal(mu, sigma) is capped at mu from above, E[X_capped] = mu - sigma * phi(0) / (1 - Phi(0))
# = mu - sigma * (1/sqrt(2pi)) / 0.5 = mu - sigma * sqrt(2/pi)
import math

sigma = assumed_mkt * variance_3
downward_bias = sigma * math.sqrt(
    2 / math.pi
)  # approximate expected bias from one-sided cap
print("\nTheoretical downward bias from one-sided cap:")
print(
    f"  E[rate bias] ≈ −{downward_bias:.2f}%  (≈ {downward_bias/assumed_mkt*100:.0f}% of assumed rate)"
)
print(
    f"  So random scenarios grow as if MKT ≈ {assumed_mkt - downward_bias:.2f}% not {assumed_mkt:.2f}%"
)
print("\nConclusion: This is NOT just a mathematical quirk — the one-sided cap is a")
print("methodological bug that systematically underapplies market interest rates.")


DECOMPOSING: HOW MUCH OF THE GAP IS RATE BIAS vs VOLATILITY DRAG vs EXPENSE VARIANCE?

Expense variance (variance_2 = 0.5, std dev = 50% of mean):
  Pre-retirement avg monthly bills: $1,739
  Std dev in bills: $870

  Pre-retirement avg monthly savings: $1,360
  Std dev in savings contribution: $680

MKT interest variance (variance_3 = 1.5, std dev = 150% of mean):
  Assumed MKT rate: 6.50%
  Std dev: 9.75%
  Normal(6.5%, 9.75%) → ~50% of draws could be negative or zero!

Theoretical downward bias from one-sided cap:
  E[rate bias] ≈ −7.78%  (≈ 120% of assumed rate)
  So random scenarios grow as if MKT ≈ -1.28% not 6.50%

Conclusion: This is NOT just a mathematical quirk — the one-sided cap is a
methodological bug that systematically underapplies market interest rates.


## Step 10: Verify Fix - Regenerate Scenarios After Removing One-Sided Cap

In [15]:
import importlib
import pages.random_scenario as rs_module

importlib.reload(rs_module)
from pages.random_scenario import RandomScenario as RandomScenarioFixed

print("=" * 80)
print("AFTER FIX: Symmetric rate distribution (one-sided cap removed)")
print("=" * 80)

# Regenerate scenarios with the fixed code
random_dfs_fixed = {}
for i in range(num_scenarios):
    print(f"\rGenerating scenario {i+1}/{num_scenarios}...", end="", flush=True)
    scenario = RandomScenarioFixed(base_scenario)
    random_dfs_fixed[i + 1] = scenario.create_full_df()
print(" ✓ Done")

# Check actual mean applied rates
all_mkt_means_fixed = [
    random_dfs_fixed[s]["var_yearly_mkt_interest"].mean() * 100
    for s in range(1, num_scenarios + 1)
]
all_rf_means_fixed = [
    random_dfs_fixed[s]["var_yearly_rf_interest"].mean() * 100
    for s in range(1, num_scenarios + 1)
]

print("\nRate accuracy after fix:")
print(
    f"  MKT assumed={assumed_mkt:.2f}%, avg actual={np.mean(all_mkt_means_fixed):.3f}%, avg bias={np.mean(all_mkt_means_fixed)-assumed_mkt:+.3f}%"
)
print(
    f"  RF  assumed={assumed_rf:.2f}%, avg actual={np.mean(all_rf_means_fixed):.3f}%, avg bias={np.mean(all_rf_means_fixed)-assumed_rf:+.3f}%"
)

# Divergence from base at key ages
print("\n\nDivergence from base scenario after fix (% total wealth):")
summary_rows_fixed = []
for target_age in [60, 75, 80, 90, 110]:
    if target_age in base_milestones:
        base_total = base_milestones[target_age]["total"]
        pct_diffs = []
        for s in range(1, num_scenarios + 1):
            rows = random_dfs_fixed[s][random_dfs_fixed[s]["age_yrs"] == target_age]
            if len(rows) > 0:
                r = rows.iloc[0]
                total = (
                    r["var_savings_account"]
                    + r["var_traditional_401k"]
                    + r["var_roth_401k"]
                    + r["var_roth_ira"]
                    + r["var_hsa"]
                )
                pct_diffs.append((total - base_total) / abs(base_total) * 100)
        if pct_diffs:
            print(
                f"  Age {target_age:3d}: avg={np.mean(pct_diffs):+6.1f}%, range=[{min(pct_diffs):+6.1f}%, {max(pct_diffs):+6.1f}%]"
            )

AFTER FIX: Symmetric rate distribution (one-sided cap removed)
Generating scenario 10/10... ✓ Done

Rate accuracy after fix:
  MKT assumed=6.50%, avg actual=5.507%, avg bias=-0.993%
  RF  assumed=3.25%, avg actual=3.403%, avg bias=+0.153%


Divergence from base scenario after fix (% total wealth):
  Age  60: avg=  -4.1%, range=[  -8.1%,   +0.4%]
  Age  75: avg= -10.8%, range=[ -23.2%,   -1.5%]
  Age  80: avg= -10.6%, range=[ -22.7%,   +5.0%]
  Age  90: avg= -13.1%, range=[ -26.6%,   +0.1%]
  Age 110: avg= -16.4%, range=[ -44.6%,   +2.2%]
